# 📦 PhonePe — Export PostgreSQL Tables to CSV


## 📋 Steps Overview
1. Connect to your local PostgreSQL (`phonepe_db`)
2. Export all 9 tables as `.csv` files into a folder called `phonepe_csv_data/`
3. Verify all files exported correctly
4. Upload the folder to Google Drive
5. Open your main notebook in Colab and update the file paths


---
## Step 1 — Import Libraries & Connect to PostgreSQL

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine

from dotenv import load_dotenv
import os

load_dotenv()

DB_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine=create_engine(DB_URL)

with engine.connect() as conn:
    print('Connected to DB successfully')

Connected to DB successfully


In [2]:
SAVE_FOLDER= "phonepe_csv_data"
os.makedirs(SAVE_FOLDER,exist_ok=True)

print(f'Folder ready: {os.path.abspath(SAVE_FOLDER)}')

Folder ready: C:\Users\Dell\Desktop\INTERNSHIPS\Labmentix(DS_AI_ML)\Mini Project\1\notebooks\phonepe_csv_data


---
## Step 3 — Export All 9 Tables to CSV`

> This cell reads each table from PostgreSQL and saves it as a `.csv` file.
> It prints the row count and file size for each table so you can verify the export.

| Table | Description |
|---|---|
| `aggregated_transaction` | State-level transaction data by type, year, quarter |
| `aggregated_user` | Device brand-level user data |
| `aggregated_insurance` | State-level insurance data |
| `map_transaction` | District-level transaction data |
| `map_user` | District-level user data (registered users, app opens) |
| `map_insurance` | District-level insurance data |
| `top_transaction` | Top districts and pincodes by transaction |
| `top_user` | Top districts and pincodes by registered users |
| `top_insurance` | Top districts and pincodes by insurance |


In [3]:
# ── All 9 table names ────────────────────────────────────────────────────
tables = [
    'aggregated_transaction',
    'aggregated_user',
    'aggregated_insurance',
    'map_transaction',
    'map_user',
    'map_insurance',
    'top_transaction',
    'top_user',
    'top_insurance',
]

# ── Export each table ─────────────────────────────────────────────────────
print(f"{'Table':<30} {'Rows':>8}  {'File Size':>12}  Status")
print('-' * 65)

for table in tables:
    # Read from PostgreSQL
    df = pd.read_sql(f'SELECT * FROM {table}', engine)

    # Save to CSV
    filepath = os.path.join(SAVE_FOLDER, f'{table}.csv')
    df.to_csv(filepath, index=False)

    # Get file size
    size_kb = os.path.getsize(filepath) / 1024
    size_str = f'{size_kb:.1f} KB' if size_kb < 1024 else f'{size_kb/1024:.2f} MB'

    print(f'{table:<30} {len(df):>8,}  {size_str:>12}  ✅')

print(f'\n🎉 All {len(tables)} tables exported successfully!')

Table                              Rows     File Size  Status
-----------------------------------------------------------------
aggregated_transaction            5,034      302.5 KB  ✅
aggregated_user                   6,732      346.0 KB  ✅
aggregated_insurance                682       29.7 KB  ✅
map_transaction                  20,604       1.22 MB  ✅
map_user                         20,608       1.01 MB  ✅
map_insurance                    13,876      683.6 KB  ✅
top_transaction                  18,295       1.05 MB  ✅
top_user                         18,296      753.5 KB  ✅
top_insurance                    12,276      580.1 KB  ✅

🎉 All 9 tables exported successfully!


---
## Step 4 — Verify All CSV Files

> This cell confirms all 9 CSV files exist in the folder and previews the first few rows of each.
> Check that row counts match what you saw in the ETL verification step.

In [5]:
# ── List all exported CSV files ──────────────────────────────────────────
print('📂 Files in phonepe_csv_data/\n')
csv_files = [f for f in os.listdir(SAVE_FOLDER) if f.endswith('.csv')]

print(f"{'File':<40} {'Size':>10}")
print('-' * 52)
for f in sorted(csv_files):
    size_kb = os.path.getsize(os.path.join(SAVE_FOLDER, f)) / 1024
    size_str = f'{size_kb:.1f} KB' if size_kb < 1024 else f'{size_kb/1024:.2f} MB'
    print(f'{f:<40} {size_str:>10}')

print(f'\nTotal files: {len(csv_files)} / 9')
if len(csv_files) == 9:
    print('✅ All 9 tables exported correctly!')
else:
    print('⚠️ Some files are missing — rerun Step 3')

📂 Files in phonepe_csv_data/

File                                           Size
----------------------------------------------------
aggregated_insurance.csv                    29.7 KB
aggregated_transaction.csv                 302.5 KB
aggregated_user.csv                        346.0 KB
map_insurance.csv                          683.6 KB
map_transaction.csv                         1.22 MB
map_user.csv                                1.01 MB
top_insurance.csv                          580.1 KB
top_transaction.csv                         1.05 MB
top_user.csv                               753.5 KB

Total files: 9 / 9
✅ All 9 tables exported correctly!


In [6]:
# ── Preview first 3 rows of each CSV ────────────────────────────────────
for table in tables:
    filepath = os.path.join(SAVE_FOLDER, f'{table}.csv')
    df_preview = pd.read_csv(filepath)
    print(f'\n=== {table} — {len(df_preview):,} rows ===')
    display(df_preview.head(3))


=== aggregated_transaction — 5,034 rows ===


,state,year,quarter,transaction_type,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,andaman-&-nicobar-islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,andaman-&-nicobar-islands,2018,1,Merchant payments,298,4.525072e+05



=== aggregated_user — 6,732 rows ===


,state,year,quarter,brand,user_count,user_percentage
0,andaman-&-nicobar-islands,2018,1,Xiaomi,1665,0.247033
1,andaman-&-nicobar-islands,2018,1,Samsung,1445,0.214392
2,andaman-&-nicobar-islands,2018,1,Vivo,982,0.145697



=== aggregated_insurance — 682 rows ===


...

,state,year,quarter,entity_type,entity_name,transaction_count,transaction_amount
0,andaman-&-nicobar-islands,2018,1,district,south andaman,5688,1.256025e+07
1,andaman-&-nicobar-islands,2018,1,district,nicobars,528,1.139849e+06
2,andaman-&-nicobar-islands,2018,1,district,north and middle andaman,442,9.316631e+05



=== top_user — 18,296 rows ===


,state,year,quarter,entity_type,entity_name,registered_users
0,andaman-&-nicobar-islands,2018,1,district,south andaman,5846
1,andaman-&-nicobar-islands,2018,1,district,north and middle andaman,632
2,andaman-&-nicobar-islands,2018,1,district,nicobars,262



=== top_insurance — 12,276 rows ===


,state,year,quarter,entity_type,entity_name,insurance_count,insurance_amount
0,andaman-&-nicobar-islands,2020,2,district,nicobars,3,565.0
1,andaman-&-nicobar-islands,2020,2,district,south andaman,3,795.0
2,andaman-&-nicobar-islands,2020,2,pincode,744301,3,565.0
